In [ ]:
# Run once if needed:
# %pip install -q transformers==4.57.3 peft==0.14.0 huggingface_hub==0.36.2
# %pip install -q accelerate==1.10.1 bitsandbytes==0.47.0 safetensors==0.6.2
# %pip install -q evaluate sacrebleu rouge_score torchao==0.9.0

In [ ]:
# ============================================================
# Local Qwen2.5-Coder-1.5B Commit Message Evaluation (Windows)
# Compares base model vs. fine-tuned LoRA adapter.
# Compatible stack: torch 2.6+cu124, transformers 4.57.3,
#                   peft 0.14.0, bitsandbytes 0.47.0
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import json
import re
import time
import warnings
warnings.filterwarnings("ignore")
from collections import Counter
from pathlib import Path

import evaluate
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")
import transformers, peft
print("transformers:", transformers.__version__, "| peft:", peft.__version__)

In [ ]:
# -------------------- Configuration --------------------

# Base model — must match adapter_config.json base_model_name_or_path
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

PROJECT_DIR = Path(r"C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand")
TEST_FILE   = PROJECT_DIR / "data" / "processed" / "test.jsonl"
ADAPTER_DIR = PROJECT_DIR / "commit-model-lora"
OUT_DIR     = PROJECT_DIR / "evaluation_qwen3b"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LIMIT          = 200   # None = full test set (69990 rows)
BATCH_SIZE     = 1     # RTX 3070 Ti Laptop (8 GB VRAM) with 4-bit: start with 1
SEQ_LEN        = 768   # must match training sequence length
MAX_NEW_TOKENS = 40

# This matches PROMPT_TEMPLATE in src/commit_model/diff_utils.py
PROMPT_TEMPLATE = "Write a Conventional Commits message for this diff.\n\n{diff}\n\nCommit message:"

In [ ]:
# -------------------- Checks --------------------

if not TEST_FILE.exists():
    raise FileNotFoundError(f"Test file not found: {TEST_FILE}")

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Adapter config not found: {ADAPTER_DIR / 'adapter_config.json'}"
    )

if not (ADAPTER_DIR / "adapter_model.safetensors").exists():
    raise FileNotFoundError(
        f"Adapter weights not found: {ADAPTER_DIR / 'adapter_model.safetensors'}\n"
        "The weights were extracted from new_weights_to_add_to_frozen_weights/commit-model-lora.zip."
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Install a CUDA-enabled PyTorch build.")

print("Test file:", TEST_FILE)
print("Adapter:", ADAPTER_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
# -------------------- Load tokenizer --------------------
# IMPORTANT: load tokenizer from base model name, NOT from ADAPTER_DIR.
# The adapter's tokenizer_config.json was saved with a newer PEFT version that
# stores extra_special_tokens as a list; transformers 4.57.3 expects a dict.
# Loading from the base model name is identical in practice.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

template_tokens = len(
    tokenizer(PROMPT_TEMPLATE.format(diff=""), add_special_tokens=False).input_ids
)
DIFF_BUDGET = max(SEQ_LEN - template_tokens - MAX_NEW_TOKENS - 8, 0)
print(f"Tokenizer: {tokenizer.__class__.__name__}, vocab={tokenizer.vocab_size}")
print(f"Diff token budget: {DIFF_BUDGET}")


def fit_prompt(diff):
    diff_ids = tokenizer(diff, add_special_tokens=False).input_ids[:DIFF_BUDGET]
    truncated_diff = tokenizer.decode(diff_ids)
    return PROMPT_TEMPLATE.format(diff=truncated_diff)

In [ ]:
# -------------------- Read test data --------------------

with open(TEST_FILE, "r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

if LIMIT is not None:
    examples = examples[:LIMIT]

if not examples:
    raise ValueError("The test file contains no examples.")

print(f"Test examples: {len(examples)}")
print(f"Sample: {examples[0]['message']}")

In [ ]:
# -------------------- Model loading and generation --------------------

def load_model(finetuned=False):
    print("\nLoading", "fine-tuned" if finetuned else "base", "model...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    if finetuned:
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
        # Hard check: confirm LoRA weights loaded, never silently fall back to base
        lora_params = [n for n, _ in model.named_parameters() if "lora_" in n]
        if not lora_params:
            raise RuntimeError(
                "CRITICAL: PeftModel loaded but no lora_ parameters found. "
                "This would evaluate the base model, not the fine-tuned adapter."
            )
        print(f"LoRA adapter confirmed: {len(lora_params)} lora_ parameter tensors.")
    model.eval()
    return model


def generate_predictions(finetuned, output_file):
    model = load_model(finetuned)
    # With device_map="auto" use the first parameter's device for inputs
    device = next(model.parameters()).device
    records = []
    try:
        with open(output_file, "w", encoding="utf-8") as out:
            for start in range(0, len(examples), BATCH_SIZE):
                batch = examples[start:start + BATCH_SIZE]
                prompts = [fit_prompt(ex["diff"]) for ex in batch]
                inputs = tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=SEQ_LEN,
                ).to(device)
                with torch.inference_mode():
                    generated = model.generate(
                        **inputs,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                new_tokens = generated[:, inputs["input_ids"].shape[1]:]
                texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
                for ex, text in zip(batch, texts):
                    pred = (text.strip().splitlines() or [""])[0].strip()
                    record = {"pred": pred, "ref": ex["message"]}
                    records.append(record)
                    out.write(json.dumps(record, ensure_ascii=False) + "\n")
                out.flush()
                print(f"{output_file.name}: {start + len(batch)}/{len(examples)}")
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return records

In [ ]:
# -------------------- Evaluate both models --------------------

started = time.time()

base_records = generate_predictions(
    finetuned=False,
    output_file=OUT_DIR / "preds_base.jsonl",
)

finetuned_records = generate_predictions(
    finetuned=True,
    output_file=OUT_DIR / "preds_finetuned.jsonl",
)

print(f"Generation completed in {(time.time() - started) / 60:.1f} minutes.")

In [ ]:
# -------------------- Metrics --------------------

TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")
VALID_RE = re.compile(
    r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)"
    r"(\([\w./-]+\))?!?:\s+\S"
)


def extract_type(message):
    match = TYPE_RE.match(message.strip())
    return match.group(1) if match else None


bleu  = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

preds = {
    "base":      base_records,
    "finetuned": finetuned_records,
}

refs       = [record["ref"] for record in base_records]
ref_types  = [extract_type(ref) for ref in refs]

majority_type, majority_count = Counter(ref_types).most_common(1)[0]

rows = [{
    "model":        f"always '{majority_type}:'",
    "valid_format": None,
    "type_acc":     majority_count / len(refs),
    "BLEU":         None,
    "ROUGE-L":      None,
}]

for name, records in preds.items():
    predictions = [record["pred"] for record in records]
    rows.append({
        "model":        name,
        "valid_format": sum(bool(VALID_RE.match(p)) for p in predictions) / len(predictions),
        "type_acc":     sum(
            extract_type(p) == t for p, t in zip(predictions, ref_types)
        ) / len(predictions),
        "BLEU": bleu.compute(
            predictions=predictions,
            references=[[ref] for ref in refs],
        )["score"],
        "ROUGE-L": rouge.compute(
            predictions=predictions,
            references=refs,
        )["rougeL"],
    })

results = pd.DataFrame(rows).set_index("model")

print(f"\nExamples: {len(refs)}\n")
print(results.to_string(float_format=lambda value: f"{value:.3f}", na_rep="-"))

results.to_json(OUT_DIR / "results.json", orient="index", indent=2)

In [ ]:
# -------------------- Per-type accuracy --------------------

print("\nPer-type accuracy (fine-tuned):")
per_type = Counter(ref_types)

for commit_type, count in per_type.most_common():
    hits = sum(
        extract_type(record["pred"]) == commit_type
        for record, reference_type in zip(finetuned_records, ref_types)
        if reference_type == commit_type
    )
    print(f"  {commit_type:<9} {hits / count:6.1%}  (n={count})")

In [ ]:
# -------------------- Sample predictions --------------------

print("\nSample predictions:")
for i in range(min(10, len(refs))):
    print(f"  ref:       {refs[i]}")
    print(f"  base:      {base_records[i]['pred']}")
    print(f"  finetuned: {finetuned_records[i]['pred']}\n")

print("All outputs saved to:", OUT_DIR)